In [24]:
from dotenv import load_dotenv
load_dotenv()

from google import genai
from google.genai import types
import os

In [3]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [4]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [5]:
llm('hey, whats up')

'Hey! Not much, just [insert a common, low-effort activity like \'relaxing,\' \'working,\' \'catching up on things\']. What\'s up with you?\n\nOr, you could be more specific:\n\n*   "Hey! Not much, just enjoying some coffee. You?"\n*   "Hey there! All good here, just getting some work done. What\'s new with you?"\n*   "Not much, just chilling. How about you?"'

In [6]:
question = "I just discovered the course. Can I join now?"

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [9]:
answer = llm(question)
print(answer)

That's great you're interested! To give you the most accurate answer, I'll need to know **which specific course you're referring to.**

Many courses offer different enrollment options:

*   **Self-paced courses:** You can often join these immediately and start whenever you like.
*   **Cohort-based or live courses:** These usually have fixed start dates. If a current session has already begun, you might need to wait for the next scheduled cohort.
*   **Courses with rolling enrollment:** Some courses allow you to join a bit late and catch up.

**Could you please tell me the name of the course you're interested in?** Once I have that, I can check its enrollment status, start date, and any specific deadlines for you!


In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [16]:
search_results = search(question)

In [17]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [19]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [20]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [21]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [22]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [25]:
# Construct the stateless message history structure
message_history = [
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text=prompt)]
    )
]

# Configure system instructions (developer role equivalent)
config = types.GenerateContentConfig(
    system_instruction=INSTRUCTIONS,
    temperature=0.7 # Optional parameter
)

In [26]:
# Execute the request
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=config
)

print(response.text)

Yes, you can join now. You can start whenever you want as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can only get a certificate if you finish the course with a "live" cohort, as you need to peer-review projects at the time the course is running.


In [31]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=80,
  prompt_token_count=568,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=568
    ),
  ],
  thoughts_token_count=311,
  total_token_count=959
)

In [39]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    # Construct the stateless message history structure
    message_history = [
        types.Content(
            role="user", 
            parts=[types.Part.from_text(text=user_prompt)]
        )
    ]

    # Configure system instructions (developer role equivalent)
    config = types.GenerateContentConfig(
        system_instruction=instructions,
        temperature=0.7 # Optional parameter
    )

    # Execute the request
    response = client.models.generate_content(
        model=model,
        contents=message_history,
        config=config
    )

    print(response.text)

In [40]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.
None


In [42]:
rag("How do I get a certificate?")

To get a certificate, you need to:

1.  **Finish the course with a "live" cohort.** Certificates are not awarded for the self-paced mode.
2.  **Pass the Capstone project.**
3.  **Peer-review 3 capstone projects** after submitting your own. This can only be done while the course is running.
4.  **Ensure your official name** (as in your identification documents) is entered in the "Second field" of your course profile, as this name will appear on your certificate.
